# 03 — Train DeepScalper
Trains one `DuelingQNetwork` per ticker using **Double DQN + Prioritized Experience Replay**.

**Training loop per ticker:**
- Up to `MAX_EPISODES=200` episodes (each episode = 1 random training day).
- Early stopping if validation Sharpe does not improve for `PATIENCE=20` episodes.
- Best model (by val Sharpe) is saved as `{TICKER}.pth`.

**Input:**  `/content/drive/MyDrive/algo_trader/data/features/{TICKER}_train.npz`  
**Output:** `/content/drive/MyDrive/algo_trader/weights/{TICKER}.pth`

> Enable GPU runtime: **Runtime → Change runtime type → T4 GPU** before running.

In [ ]:
!pip install -q torch torchvision gymnasium numpy pandas pyarrow pytz tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
RAW_DIR     = '/content/drive/MyDrive/algo_trader/data/raw'
WEIGHTS_DIR = '/content/drive/MyDrive/algo_trader/weights'
os.makedirs(WEIGHTS_DIR, exist_ok=True)
print(f'Raw data dir:  {RAW_DIR}')
print(f'Weights dir:   {WEIGHTS_DIR}')

In [ ]:
REPO_URL = 'https://github.com/rohanpatrick568/deepscalper_copilot.git'
REPO_DIR = '/content/deepscalper_copilot'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

# algo_trader on path → enables `from colab.deepscalper.X import Y`
ALGO_DIR = REPO_DIR + '/algo_trader'
if ALGO_DIR not in sys.path:
    sys.path.insert(0, ALGO_DIR)
print('Repo on path ✓')

In [ ]:
import torch
print(f'PyTorch {torch.__version__}')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

# ── Architecture — must match config.py ──────────────────────────────────────
MACRO_DIM     = 11   # Table 2 macro features
LOB_DIM       = 5    # Intrabar microstructure (LOB proxy)
PRIV_DIM      = 2    # Private state: (position_flag, unrealized_pnl_pct)
N_DIR         = 3    # Direction branch: 0=HOLD 1=BUY 2=SELL
N_SIZE        = 4    # Size branch: 0=25% 1=50% 2=75% 3=100% of max_notional
GRU_HIDDEN    = 128  # GRU hidden per stream in MicroEncoder
MACRO_EMBED   = 64   # MacroEncoder MLP output dim
FC_HIDDEN     = 128  # BDQ head FC width
LOOKBACK_BARS = 60   # Observation window length T

# ── Training hyperparameters ─────────────────────────────────────────────────
MAX_EPISODES  = 200
PATIENCE      = 20   # Early stopping: episodes without val Sharpe improvement
EVAL_EVERY    = 5    # Validate every N training episodes
EVAL_EPISODES = 10   # Episodes per validation Sharpe estimate
TRAIN_FRAC    = 0.80 # Time-ordered train/val split

SP100_TICKERS = [
    'AAPL','MSFT','AMZN','NVDA','GOOGL','GOOG','META','TSLA','BRK.B','UNH',
    'LLY','JPM','V','AVGO','XOM','MA','COST','PG','JNJ','HD',
    'ABBV','ORCL','BAC','WMT','NFLX','KO','CRM','CVX','MRK','AMD',
    'CSCO','PEP','ACN','LIN','TMO','MCD','ABT','IBM','GE','TXN',
    'PM','GS','ISRG','CAT','AXP','SPGI','AMGN','RTX','PFE','BKNG',
    'DHR','MS','INTU','BLK','T','VRTX','HON','NEE','UNP','SYK',
    'C','LOW','TJX','ADP','GILD','DE','PANW','BMY','AMAT','MDT',
    'PLD','SBUX','ADI','TMUS','ETN','SCHW','CB','MMC','BA','SO',
    'MO','WFC','UPS','CI','MDLZ','DUK','CL','INTC','REGN','PH',
    'EOG','SLB','ELV','APD','MCK','COF','ZTS','BSX','GEV','CME',
]
print(f'{len(SP100_TICKERS)} tickers loaded')

In [ ]:
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

from colab.deepscalper.agent import DeepScalperAgent
from colab.deepscalper.environment import ScalperEnv
from colab.deepscalper.utils import (
    compute_macro_features,
    compute_micro_features,
    compute_day_starts,
    compute_sharpe,
)


def evaluate_agent(agent: DeepScalperAgent, env: ScalperEnv, n_episodes: int) -> float:
    """Run n_episodes greedy (ε=0) and return the annualised Sharpe ratio."""
    saved_epsilon = agent.epsilon
    agent.epsilon = 0.0
    episode_rewards = []
    for _ in range(n_episodes):
        obs, _ = env.reset()
        done = False
        total_r = 0.0
        while not done:
            dir_act, size_act = agent.select_action(obs)
            obs, r, terminated, truncated, _ = env.step(np.array([dir_act, size_act]))
            total_r += r
            done = terminated or truncated
        episode_rewards.append(total_r)
    agent.epsilon = saved_epsilon
    return compute_sharpe(episode_rewards)


# ── Main training loop ────────────────────────────────────────────────────────
training_log = []

for ticker in tqdm(SP100_TICKERS, desc='Tickers'):
    weights_path = f'{WEIGHTS_DIR}/{ticker}.pth'
    if os.path.exists(weights_path):
        print(f'{ticker}: weights already exist — skipping.')
        continue

    raw_path = f'{RAW_DIR}/{ticker}.parquet'
    if not os.path.exists(raw_path):
        print(f'WARNING: {raw_path} missing — skipping {ticker}.')
        continue

    # ── Load & compute features ───────────────────────────────────────────────
    bars = pd.read_parquet(raw_path)
    bars.columns = [c.lower() for c in bars.columns]
    bars = bars[['open', 'high', 'low', 'close', 'volume']].astype(float)

    macro_feats = compute_macro_features(bars)          # (n_bars, 11)
    lob_feats   = compute_micro_features(bars)          # (n_bars, 5)
    close_arr   = bars['close'].values.astype(np.float64)
    day_starts  = compute_day_starts(bars.index)        # bar-level indices

    if len(day_starts) < 7:
        print(f'WARNING: {ticker} has only {len(day_starts)} days — skipping.')
        continue

    # ── Time-ordered 80/20 split ──────────────────────────────────────────────
    n_bars    = len(bars)
    split_bar = int(n_bars * TRAIN_FRAC)

    train_day_starts = [d for d in day_starts if d < split_bar]
    # Shift val day indices so they're relative to the val slice start
    val_day_starts   = [d - split_bar for d in day_starts if d >= split_bar]

    if len(train_day_starts) < 5 or len(val_day_starts) < 2:
        print(f'WARNING: {ticker} insufficient days after split — skipping.')
        continue

    # ── Environments ──────────────────────────────────────────────────────────
    train_env = ScalperEnv(
        lob_features   = lob_feats[:split_bar],
        macro_features = macro_feats[:split_bar],
        close_prices   = close_arr[:split_bar],
        day_starts     = train_day_starts,
        lookback_bars  = LOOKBACK_BARS,
    )
    val_env = ScalperEnv(
        lob_features   = lob_feats[split_bar:],
        macro_features = macro_feats[split_bar:],
        close_prices   = close_arr[split_bar:],
        day_starts     = val_day_starts,
        lookback_bars  = LOOKBACK_BARS,
    )

    # ── Agent ─────────────────────────────────────────────────────────────────
    agent = DeepScalperAgent(
        macro_dim   = MACRO_DIM,
        lob_dim     = LOB_DIM,
        priv_dim    = PRIV_DIM,
        n_dir       = N_DIR,
        n_size      = N_SIZE,
        gru_hidden  = GRU_HIDDEN,
        macro_embed = MACRO_EMBED,
        fc_hidden   = FC_HIDDEN,
        device      = DEVICE,
    )

    best_sharpe    = -np.inf
    patience_count = 0

    for episode in range(1, MAX_EPISODES + 1):
        # ── Training episode ──────────────────────────────────────────────────
        obs, _ = train_env.reset()
        done = False
        while not done:
            dir_act, size_act = agent.select_action(obs)
            next_obs, reward, terminated, truncated, info = train_env.step(
                np.array([dir_act, size_act])
            )
            done = terminated or truncated

            # Apply hindsight bonus (Section 4.2) using future price from info
            h_reward = agent.compute_hindsight_reward(
                base_reward   = reward,
                position      = info['position'],
                current_price = info['current_price'],
                future_price  = info['future_price'],
            )

            agent.store(
                obs         = obs,
                dir_action  = dir_act,
                size_action = size_act,
                reward      = h_reward,
                next_obs    = next_obs,
                done        = done,
                vol_target  = info['vol_target'],
            )
            agent.learn()
            obs = next_obs

        # ── Periodic validation ───────────────────────────────────────────────
        if episode % EVAL_EVERY == 0:
            val_sharpe = evaluate_agent(agent, val_env, EVAL_EPISODES)

            if val_sharpe > best_sharpe:
                best_sharpe    = val_sharpe
                patience_count = 0
                agent.save(weights_path)
            else:
                patience_count += 1

            if patience_count >= PATIENCE:
                print(f'{ticker}: early stop at ep {episode} (best Sharpe={best_sharpe:.3f})')
                break

    # Save weights if no eval checkpoint was written (e.g. very short training)
    if not os.path.exists(weights_path):
        agent.save(weights_path)

    training_log.append({'ticker': ticker, 'best_val_sharpe': round(best_sharpe, 4)})
    print(f'{ticker}: best val Sharpe = {best_sharpe:.4f}')


print('\n=== TRAINING COMPLETE ===')
if training_log:
    df_log = pd.DataFrame(training_log).sort_values('best_val_sharpe', ascending=False)
    print(df_log.to_string(index=False))

log_path = '/content/drive/MyDrive/algo_trader/training_log.csv'
pd.DataFrame(training_log).to_csv(log_path, index=False)
print(f'\nTraining log saved → {log_path}')